# 3. Modelado Balanceado con Random Undersampling
**Autores:** Eduardo Estefanía Ovejero, David Blasco Tena y Alejandro Jesús Castro García

Origen de los datos: https://www.kaggle.com/datasets/pavansubhasht/ibm-hr-analytics-attrition-dataset

## Objetivo
El objetivo de este notebook es entrenar y evaluar un modelo de aprendizaje automático capaz de predecir si un empleado abandonará la empresa (`Attrition`) a partir de las variables previamente preparadas en la fase de *Feature Engineering*.

En esta fase se realizará:
- La carga de los conjuntos de entrenamiento y prueba ya transformados
- El entrenamiento de un modelo de **Regresión logistica**.
- La generación de predicciones sobre el conjunto de prueba
- La evaluación del modelo mediante métricas apropiadas para un problema de clasificación binaria con desbalanceo de clases.

## 3.1. Importación de librerias y carga de datos
Se cargan los conjuntos de entrenamiento y prueba generados en el notebook anterior.
Estos datos ya han sido codificados y escalados, por lo que están listos para la fase de modelado.

In [1]:
import pandas as pd
import warnings
import numpy as np

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

In [2]:
X_train = pd.read_csv("X_train_escalado.csv")
X_test = pd.read_csv("X_test_escalado.csv")
y_train = pd.read_csv("y_train.csv").squeeze()
y_test = pd.read_csv("y_test.csv").squeeze()

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (1176, 39)
X_test: (294, 39)
y_train: (1176,)
y_test: (294,)


# 3.2. Elección de Métricas de Evaluación

## Problemática del Desbalance de Clases

En problemas de abandono empresarial, típicamente existe un **fuerte desbalance**:
- La mayoría de empleados **NO abandonan** (clase mayoritaria)
- Solo una minoría **SÍ abandona** (clase minoritaria - nuestra clase de interés)

### ¿Por qué NO usar Accuracy convencional?

La **accuracy** simple puede ser engañosa en datos desbalanceados:

**Ejemplo:** Si solo el 15% de empleados abandonan:
- Un modelo que **siempre prediga "No abandona"** obtendría 85% de accuracy
- Pero sería **completamente inútil** para detectar empleados en riesgo

### Métrica Principal: Balanced Accuracy

**Balanced Accuracy** = (Sensibilidad + Especificidad) / 2

Promedia el rendimiento en **ambas clases**, por lo que:
- No se ve sesgada por el desbalance
- Penaliza modelos que ignoran la clase minoritaria
- Valora equitativamente la detección de ambos grupos

### Métricas Complementarias

1. **Precision (abandono):** De los que predecimos que abandonan, ¿cuántos realmente lo hacen?
   - Alta precisión → Pocas falsas alarmas
   - Importante para no desperdiciar recursos en intervenciones innecesarias

2. **Recall (abandono):** De los que realmente abandonan, ¿cuántos detectamos?
   - Alto recall → Detectamos la mayoría de casos de riesgo
   - Crítico para no perder empleados valiosos

3. **F1-Score (abandono):** Media armónica de precision y recall
   - Balance entre ambas métricas
   - Útil cuando queremos optimizar ambas simultáneamente

4. **Curva ROC y AUC (Area Under the Curve):** La curva ROC visualiza el compromiso (*trade-off*) entre la tasa de verdaderos positivos (Recall) y la tasa de falsos positivos en todos los umbrales de probabilidad posibles.
   - **AUC:** Es un valor único entre 0 y 1 que resume el rendimiento global del modelo. Un AUC de 0.5 equivale a predicciones aleatorias, mientras que valores más cercanos a 1.0 indican una excelente capacidad del modelo para distinguir correctamente entre los empleados que se quedan y los que abandonan, independientemente del umbral elegido.

## 3.3. Entenamiento de los modelos


In [3]:
def activation_function_sigmoid(x):
    return 1. / (1. + np.exp(-x))

def computation_graph_sigmoid(x ,w, b):

    return activation_function_sigmoid(w * x + b)

In [4]:
def bce_loss_function(t,y, clip = None,*args):
    idx_0 = t==0
    idx_1 = ~idx_0

    loss = np.zeros(y.shape)

    loss[idx_1] = np.log(y[idx_1])
    loss[idx_0] = np.log(1-y[idx_0])

    ## add +- 1e12 for numerical stability on infs
    if isinstance(clip,float):
        loss[loss == np.inf] = clip
        loss[loss == -np.inf] = -clip

    return -1*loss

In [5]:
def grad_bce_loss_wrt_sigmoid_model_multivariable(x, t, w, b):
    """
    Applies chain rule. Adaptado para múltiples variables (matrices).
    x: Matriz de datos (ej. 1470 filas, 39 columnas)
    t: Vector objetivo real (ej. 1470 valores de 0 o 1)
    w: Vector de pesos (39 pesos, uno para cada columna)
    b: Sesgo (bias), un solo número
    """
    ## forward operation
    z = np.dot(x, w) + b
    y = activation_function_sigmoid(z)

    ## backward operation (compute gradients / backpropagation)
    idx_0 = (t == 0)
    idx_1 = ~idx_0

    dL_dy = np.zeros(y.shape)

    # dL/dy. 1e-12 is added/substracted for stability on +-inf values
    dL_dy[idx_0] = - 1/(1 - y[idx_0] + 1e-12)
    dL_dy[idx_1] = 1/(y[idx_1] + 1e-12)

    dL_dy *= -1

    # dy/dz
    dy_dz = activation_function_sigmoid(z) * (1 - activation_function_sigmoid(z))

    # Multiplicamos el error por cada una de las 39 columnas de 'x'
    grad_w = np.dot(x.T, dL_dy * dy_dz) 
    grad_b = np.sum(dL_dy * dy_dz)

    return grad_w, grad_b

In [6]:
df_train = pd.concat([X_train, y_train], axis=1)

# 1. Separamos a los que renuncian y los que se quedan
df_abandona = df_train[df_train['Attrition'] == 1]
df_sequeda = df_train[df_train['Attrition'] == 0]

# 2. Vemos cuántos abandonan (la minoría)
num_abandonos = len(df_abandona)

# 3. Submuestreamos aleatoriamente a los que se quedan para igualar cantidades
df_sequeda_reducido = df_sequeda.sample(n=num_abandonos, random_state=42)

# 4. Juntamos y barajamos (shuffle) el nuevo dataset balanceado (50/50)
df_balanceado = pd.concat([df_abandona, df_sequeda_reducido]).sample(frac=1, random_state=42)

# 5. Volvemos a separar en X_train e y_train
X_train_bal = df_balanceado.drop('Attrition', axis=1)
y_train_bal = df_balanceado['Attrition']

print(f"Tamaño del dataset balanceado: {len(df_balanceado)} filas")

Tamaño del dataset balanceado: 380 filas


In [7]:
def entrenar_regresion_logistica(X, y, learning_rate=0.001, epochs=1000):
    """
    X: Datos de entrenamiento
    y: Target de entrenamiento
    """
    # Convertimos a numpy por si vienen en formato Pandas
    X_np = X.values if isinstance(X, pd.DataFrame) else X
    y_np = y.values.reshape(-1, 1) if isinstance(y, pd.Series) else y.reshape(-1, 1)
    
    m, n = X_np.shape
    
    # 1. Inicializamos los 39 pesos (w) y el sesgo (b) a ceros
    w = np.zeros((n, 1))
    b = 0.0
    
    # 2. Bucle de aprendizaje
    for i in range(epochs):
        # Llamamos a la función para calcular los gradientes
        grad_w, grad_b = grad_bce_loss_wrt_sigmoid_model_multivariable(X_np, y_np, w, b)
        
        # 3. Actualizamos los pesos restando el gradiente * tasa de aprendizaje
        w = w - learning_rate * grad_w
        b = b - learning_rate * grad_b
        
        # Imprimimos progreso
        if (i+1) % 200 == 0:
            print(f"Iteración {i+1}/{epochs} completada...")
            
    print("¡Entrenamiento finalizado!")
    return w, b

w_final, b_final = entrenar_regresion_logistica(X_train_bal, y_train_bal, learning_rate=0.001, epochs=1000)

Iteración 200/1000 completada...
Iteración 400/1000 completada...
Iteración 600/1000 completada...
Iteración 800/1000 completada...
Iteración 1000/1000 completada...
¡Entrenamiento finalizado!


In [8]:
print(f'Obtained parameters: w = {w_final}, b = {b_final}')

Obtained parameters: w = [[-0.06922128]
 [ 0.90273037]
 [-0.28288369]
 [ 0.41289279]
 [ 0.07492601]
 [-0.44726966]
 [ 0.4072207 ]
 [-0.11757989]
 [-0.17580885]
 [-0.49474368]
 [ 0.12360748]
 [ 0.2205096 ]
 [ 0.51916833]
 [ 1.60635478]
 [-0.07450935]
 [-0.30917416]
 [-0.12111309]
 [-0.77048239]
 [-0.14957324]
 [-0.23793064]
 [-0.22807408]
 [ 0.30996051]
 [-1.34142963]
 [ 0.09053456]
 [-0.58342385]
 [ 0.03143595]
 [-0.30899792]
 [-1.58154625]
 [ 0.67841307]
 [-0.29275654]
 [ 1.33888016]
 [ 0.03095645]
 [ 0.11621718]
 [-1.02761621]
 [-0.21593302]
 [-0.7453114 ]
 [ 0.46659065]
 [ 0.59962896]
 [ 1.13275038]], b = -1.9483063265777878


## 3.4. Evaluación del Modelo e Impacto Empresarial

La evaluación permite identificar:
- Cuántas fugas de talento reales somos capaces de predecir y prevenir a tiempo.
- El nivel de fiabilidad (tasa de falsas alarmas) al que se enfrenta RRHH cuando el modelo alerta sobre un empleado en riesgo.
- El balance costo-beneficio de implementar políticas de retención proactivas basadas en el algoritmo.
- El límite o "techo" predecible del comportamiento humano utilizando la información corporativa actual.

In [9]:
def predecir(X, w, b, umbral=0.5):
    # Convertimos a numpy
    X_np = X.values if isinstance(X, pd.DataFrame) else X
    
    # Calculamos la fórmula lineal: z = X*w + b
    z = np.dot(X_np, w) + b
    
    # Lo pasamos por la sigmoide para obtener probabilidades (0 a 1)
    probabilidades = activation_function_sigmoid(z)
    
    # Si la probabilidad es mayor al umbral (50%), predecimos 1 (Abandona)
    predicciones = (probabilidades >= umbral).astype(int)
    
    return predicciones

# Hacemos las predicciones en nuestro examen final (Test)
y_pred = predecir(X_test, w_final, b_final)

In [10]:
def evaluar_modelo(y_real, y_pred, y_prob):
    """
    y_real: Los valores reales (0 o 1)
    y_pred: Las predicciones del modelo (0 o 1, tras aplicar el umbral)
    y_prob: Las probabilidades en bruto (0.0 a 1.0) antes del umbral
    """
    # Aseguramos que sean arrays de numpy de 1 dimensión
    y_real = np.array(y_real).flatten()
    y_pred = np.array(y_pred).flatten()
    y_prob = np.array(y_prob).flatten()

    # 1. MATRIZ DE CONFUSIÓN (Contar coincidencias)
    TP = np.sum((y_real == 1) & (y_pred == 1))
    TN = np.sum((y_real == 0) & (y_pred == 0))
    FP = np.sum((y_real == 0) & (y_pred == 1))
    FN = np.sum((y_real == 1) & (y_pred == 0))
    
    matriz_confusion = np.array([[TN, FP], 
                                 [FN, TP]])
    
    # 2. BALANCED ACCURACY
    # Sensibilidad (Recall): De los que realmente abandonan, ¿cuántos acertamos?
    # Sumamos 1e-12 para evitar el error matemático de dividir entre cero
    sensibilidad = TP / (TP + FN + 1e-12)
    
    # Especificidad: De los que realmente se quedan, ¿cuántos acertamos?
    especificidad = TN / (TN + FP + 1e-12)
    
    balanced_accuracy = (sensibilidad + especificidad) / 2
    
    # 3. F1-SCORE
    # Precision: De los que dijimos que abandonan, ¿cuántos realmente lo hicieron?
    precision = TP / (TP + FP + 1e-12)
    
    # El F1 es la media armónica entre Precision y Recall (Sensibilidad)
    f1_score = 2 * (precision * sensibilidad) / (precision + sensibilidad + 1e-12)
    
    # 4. ROC AUC (Método de comparaciones por pares)
    # Separamos las probabilidades de los que se fueron y los que se quedaron
    prob_pos = y_prob[y_real == 1]
    prob_neg = y_prob[y_real == 0]
    
    pares_correctos = 0
    # Comparamos todos contra todos
    for p in prob_pos:
        for n in prob_neg:
            if p > n:        # El modelo le dio más probabilidad al que se fue (¡Bien!)
                pares_correctos += 1
            elif p == n:     # El modelo les dio la misma probabilidad (Empate)
                pares_correctos += 0.5
                
    auc = pares_correctos / (len(prob_pos) * len(prob_neg) + 1e-12)
    
    # --- IMPRESIÓN DE RESULTADOS ---
    print("=== MATRIZ DE CONFUSIÓN ===")
    print(f"                   Predicción")
    print(f"                No (0)   Sí (1)")
    print(f"Real No (0) |   {TN:4d}      {FP:4d}")
    print(f"Real Sí (1) |   {FN:4d}      {TP:4d}\n")
    
    print("=== MÉTRICAS DE RENDIMIENTO ===")
    print(f"Balanced Accuracy: {balanced_accuracy * 100:.2f}%")
    print(f"Precision:         {precision * 100:.2f}%")
    print(f"Recall (Sensib):   {sensibilidad * 100:.2f}%")
    print(f"Especificidad:     {especificidad * 100:.2f}%")
    print(f"F1-Score:          {f1_score * 100:.2f}%")
    print(f"ROC AUC:           {auc:.4f}")
    
    return matriz_confusion, balanced_accuracy, f1_score, auc

In [11]:
# 1. Modificamos un poco la función predecir de antes para que también devuelva probabilidades
def predecir_con_probabilidad(X, w, b, umbral=0.5):
    X_np = X.values if isinstance(X, pd.DataFrame) else X
    z = np.dot(X_np, w) + b
    probabilidades = activation_function_sigmoid(z)
    predicciones = (probabilidades >= umbral).astype(int)
    return predicciones, probabilidades

# 2. Obtenemos las predicciones y las probabilidades para el Test
y_pred, y_prob = predecir_con_probabilidad(X_test, w_final, b_final, umbral=0.5)

# 3. Evaluamos
cm, b_acc, f1, auc = evaluar_modelo(y_test, y_pred, y_prob)

=== MATRIZ DE CONFUSIÓN ===
                   Predicción
                No (0)   Sí (1)
Real No (0) |    191        56
Real Sí (1) |     14        33

=== MÉTRICAS DE RENDIMIENTO ===
Balanced Accuracy: 73.77%
Precision:         37.08%
Recall (Sensib):   70.21%
Especificidad:     77.33%
F1-Score:          48.53%
ROC AUC:           0.7896


### Conclusiones

El modelo final (Regresión Logística con Random Undersampling) mitiga el fuerte desbalanceo de clases:

1. **Balanced Accuracy (73.77%):** El modelo demuestra una sólida capacidad de generalización, penalizando equitativamente los errores en ambas clases y evitando el sesgo de la clase mayoritaria.

2. **ROC AUC (0.7896):** Evaluado mediante comparaciones por pares, este valor indica que el 79% de las veces el modelo ordena correctamente a un empleado que abandona por encima de uno que se queda. Es una métrica excelente para datos de comportamiento humano.

3. **Recall vs Precision:** La matriz de confusión revela 33 Verdaderos Positivos frente a 14 Falsos Negativos, alcanzando un **Recall del 70.21%**. Aunque la Precisión es baja (37.08%, con 56 Falsos Positivos), este comportamiento es el óptimo para el dominio del problema. En el contexto de Recursos Humanos, maximizar la sensibilidad (atrapar la fuga de talento) justifica el coste marginal de lidiar con falsas alarmas, dado que las políticas de retención proactivas son económicamente mucho más viables que el coste de reemplazo de personal.